# 🚀 Allwinner H713 / MagCubic HY300 Pro+ Mainline Linux Colab Builder

This notebook provides a complete, interactive, high-speed build and development environment for **Allwinner H713** projectors (MagCubic HY300 Pro+, HY310, etc.) on Google Colab.

### Features:
- **Fast Multi-Core Cloud Builds**: Leverages Colab's fast CPUs and high bandwidth without local host clutter.
- **Dual-Mode Workflow**:
  1. **Full Automated 11-Step Build**: Produces the final installable `.img` and partition metadata.
  2. **Modular Component Builds**: Build and test individual targets (BL31, U-Boot, Kernel FIT, or Rootfs) in isolation.
- **Live Kernel Patching**: Modify driver source (e.g. Audio Pop fix in `sun4i-codec.c`) and rebuild the kernel FIT image in seconds without re-doing the rootfs.
- **Automatic QEMU ARM64 Emulation**: Pre-configured `binfmt_misc` kernel hooks for Debian Trixie rootfs bootstrapping.
- **Direct Export**: Download directly to your browser or save directly into Google Drive.

--- 
## 1. Optional: Mount Google Drive
Mount Google Drive to persist build logs, eMMC vendor dumps, or finished `.img` files across Colab sessions.

In [ ]:
# @title Mount Google Drive (Optional)
from google.colab import drive
import os

MOUNT_DRIVE = False # @param {type:"boolean"}
if MOUNT_DRIVE:
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/H713_Builds', exist_ok=True)
    print("Google Drive mounted. Output folder: /content/drive/MyDrive/H713_Builds")
else:
    print("Skipping Google Drive mount.")

--- 
## 2. System Dependencies & ARM64 Emulation Setup
Installs `podman`, `qemu-user-static`, and registers ARM64 `binfmt_misc` into the Colab Linux kernel so the container can execute ARM64 binaries transparently.

In [ ]:
# @title Setup Host QEMU binfmt & Podman
%%bash
set -e
echo "==> Setting up QEMU binfmt for ARM64 emulation..."
apt-get update -qq
apt-get install -y -qq podman qemu-user-static binfmt-support

# Register binfmt with the Linux kernel fix-binary flag (F)
docker run --privileged --rm tonistiigi/binfmt --install arm64

# Verify ARM64 binary execution
qemu-aarch64-static --version | head -n 1
echo "==> Host setup complete."

--- 
## 3. Clone Repository
Clones the H713 Linux workspace including all pinned submodules (`u-boot`, `arm-trusted-firmware`, `crust`, `sunxi-tools`).

In [ ]:
# @title Clone Repository with Submodules
import os

REPO_URL = "https://github.com/YuujiLab/allwinner-h713-linux.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

if not os.path.exists("/content/allwinner-h713-linux"):
    !git clone --recursive --branch {BRANCH} {REPO_URL} /content/allwinner-h713-linux
else:
    %cd /content/allwinner-h713-linux
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}
    !git submodule update --init --recursive

%cd /content/allwinner-h713-linux
!git status

--- 
## 4. Spin Up `h713-build` Podman Container & Toolchain
Initializes the official Ubuntu 24.04 build container with Clang 20, LLVM, `mmdebstrap`, `qemu-user-static`, and Debian archive keys.

In [ ]:
# @title Initialize Container & LLVM 20 Toolchain
%%bash
cd /content/allwinner-h713-linux

# Clean up any previous container
podman rm -f h713-build 2>/dev/null || true

echo "==> Launching h713-build container..."
podman run -d --name h713-build --userns=keep-id \
  -v "$PWD":/work -w /work docker.io/library/ubuntu:24.04 sleep infinity

echo "==> Installing dependencies inside container..."
podman exec -u root h713-build apt-get update -qq
podman exec -u root h713-build apt-get install -y -qq --no-install-recommends \
  clang lld llvm device-tree-compiler swig flex bison u-boot-tools \
  build-essential bc kmod libssl-dev libelf-dev python3-dev git curl \
  ca-certificates xz-utils cpio rsync libgnutls28-dev uuid-dev \
  libncurses-dev pkg-config libusb-1.0-0-dev libfdt-dev python3-capstone \
  python3-setuptools python3-pyelftools mmdebstrap e2fsprogs zstd \
  meson ninja-build python3-mako python3-yaml python3-packaging \
  libwayland-bin libwayland-dev libfreetype-dev \
  qemu-user-static debian-archive-keyring arch-test

echo "==> Installing LLVM 20 (required for U-Boot SREC and kernel build)..."
podman exec -u root h713-build bash -c '
  curl -fsSL https://apt.llvm.org/llvm-snapshot.gpg.key -o /usr/share/keyrings/llvm.asc
  echo "deb [signed-by=/usr/share/keyrings/llvm.asc] http://apt.llvm.org/noble/ llvm-toolchain-noble-20 main" \
      > /etc/apt/sources.list.d/llvm20.list
  apt-get update -qq && apt-get install -y -qq clang-20 lld-20 llvm-20
  for t in clang ld.lld llvm-objcopy llvm-ar llvm-nm llvm-objdump llvm-readelf llvm-strip; do
      ln -sf /usr/bin/$t-20 /usr/local/bin/$t
  done'

echo "==> Container ready. Toolchain versions:"
podman exec h713-build clang --version | head -n 1
podman exec h713-build ld.lld --version | head -n 1

--- 
## 5. Configuration & Target Selection
Select your projector board, variant, and rootfs packages.

In [ ]:
# @title Target Board Configuration
BOARD = "hy200-qz713df-a1" # @param ["hy200-qz713df-a1", "hy310", "l018"]
VARIANT = "projector09220931" # @param ["projector09220931", "projector05061226", "none"]
PROFILE = "media,desktop" # @param ["media,desktop", "media", "desktop", "none"]
VERSION = "v0.97-beta" # @param {type:"string"}
ENABLE_STANDBY = True # @param {type:"boolean"}

BUILD_ARGS = ["--board", BOARD, "--version", VERSION]
if VARIANT != "none":
    BUILD_ARGS += ["--variant", VARIANT]
if PROFILE != "none":
    BUILD_ARGS += ["--profile", PROFILE]
if ENABLE_STANDBY:
    BUILD_ARGS += ["--standby"]

print("Configured arguments:", " ".join(BUILD_ARGS))

--- 
## 6. Option A: Full Automated Release Build
Executes the official 11-step build script (`release/build-all.sh`). This produces a complete `.img` ready for flashing via `h713-install`.

In [ ]:
# @title Execute Full Release Build (11 Steps)
import subprocess

cmd = ["release/build-all.sh"] + BUILD_ARGS
print(f"Executing: {' '.join(cmd)}\n")

proc = subprocess.Popen(cmd, cwd="/content/allwinner-h713-linux", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode == 0:
    print("\n✅ BUILD SUCCESSFUL!")
else:
    print(f"\n❌ BUILD FAILED with exit code {proc.returncode}")

--- 
## 7. Option B: Modular / Component-by-Component Builds
Use these cells when developing, debugging, or testing specific parts without running all 11 steps.

In [ ]:
# @title Build TF-A (BL31) Only
%%bash
cd /content/allwinner-h713-linux
mainline/build/build.sh bl31

In [ ]:
# @title Build Crust Firmware Only (ARISC Standby)
%%bash
cd /content/allwinner-h713-linux
mainline/build/build.sh crust

In [ ]:
# @title Build U-Boot (SPL + Proper + Installer)
%%bash
cd /content/allwinner-h713-linux
mainline/build/uboot-build.sh mainline/build/out/uboot-release hy200_qz713df_a1 release
mainline/build/uboot-build.sh mainline/build/out/uboot-installer hy200_qz713df_a1 installer
mainline/build/uboot-build.sh mainline/build/out/uboot-probe h713_probe

In [ ]:
# @title Build Kernel & FIT Image Only (Fast incremental rebuilds)
%%bash
cd /content/allwinner-h713-linux
# Incremental kernel compilation (~30 seconds after changes)
BOARD=hy200-qz713df-a1 mainline/build/build.sh kernel
ls -lh mainline/build/out/h713-kernel.fit

--- 
## 8. Interactive Kernel Modification (e.g. Audio Codec Anti-Pop Patch)
Edit kernel driver sources or apply fixes directly, then recompile just the kernel FIT image.

In [ ]:
# @title Inspect or Edit Kernel Driver Source
CODEC_FILE = "/content/allwinner-h713-linux/mainline/external/linux/sound/soc/sunxi/sun4i-codec.c"

import os
if os.path.exists(CODEC_FILE):
    print(f"Found {CODEC_FILE}")
    # Example: Check headphone control register definitions
    !grep -n -C 5 "SUNXI_HEADPHONE_REG" {CODEC_FILE} || echo "(Not yet defined or in patches)"
else:
    print("Kernel tree not yet checked out. Run the kernel build step first.")

--- 
## 9. Inspect Build Logs & Diagnostics

In [ ]:
# @title View Step Logs
import glob
import os

LOG_DIR = "/content/allwinner-h713-linux/mainline/build/out"
logs = glob.glob(f"{LOG_DIR}/*.log")
if logs:
    print("Available logs:")
    for l in sorted(logs):
        print(f" - {os.path.basename(l)} ({os.path.getsize(l)} bytes)")
    
    # Print tail of rootfs or kernel log
    target_log = f"{LOG_DIR}/rootfs-build.log"
    if os.path.exists(target_log):
        print(f"\n--- Tail of {target_log} ---")
        !tail -n 25 {target_log}
else:
    print("No logs found yet.")

--- 
## 10. Download Output Artifacts
Download the built firmware images, bootloader binaries, and partition tables.

In [ ]:
# @title Download Artifacts
import glob
import os
import shutil
from google.colab import files

OUT_DIR = "/content/allwinner-h713-linux/installer/out"
images = glob.glob(f"{OUT_DIR}/*.img")
tables = glob.glob(f"{OUT_DIR}/*.json")

print("Finished Artifacts:")
for f in images + tables:
    print(f" - {os.path.basename(f)} ({os.path.getsize(f) / (1024*1024):.2f} MB)")

# Copy to Google Drive if mounted
if os.path.exists("/content/drive/MyDrive/H713_Builds"):
    for f in images + tables:
        dest = os.path.join("/content/drive/MyDrive/H713_Builds", os.path.basename(f))
        shutil.copy2(f, dest)
        print(f"Saved to Drive: {dest}")

# Or download directly in browser:
DOWNLOAD_NOW = False # @param {type:"boolean"}
if DOWNLOAD_NOW:
    for f in tables:
        files.download(f)
    for f in images:
        files.download(f)